# 团体座位预订背包问题 (GSR-KP)

**类别：** 装箱

来源：[https://www.hexaly.com/templates/group-seat-reservation-knapsack-problem-gsr-kp](https://www.hexaly.com/templates/group-seat-reservation-knapsack-problem-gsr-kp)


## 问题

我们定义**团体座位预订背包问题 (GSR-KP)** 如下。我们考虑一列座位数固定的火车，它需要停靠若干车站。在一组预订请求中，我们必须选择接受其中的一个子集。每个请求指定了一个团体人数和一个出行区间（从起始站到终点站）。对于每个团体，所分配的座位必须是相邻的。在整个路线上的任何位置，已占用的座位数都不能超过火车的座位容量。

目标是最大化整个行程的火车的总体利用率，以整个行程预订的座位总数来衡量。

	

### 学到的建模原则

- 添加 [optional interval decision variables](https://www.hexaly.com/docs/last/modelerreference/standardlibrary/builtinfunctions.html#optionalInterval) 来为每个被选中的请求建模所分配的连续座位
- 使用 [presence operator](https://www.hexaly.com/docs/last/modelerreference/standardlibrary/builtinfunctions.html#presence) 来检测某个请求是否被选中


## 数据

我们提供的实例来自 [Clausen et al.](https://hjemmesider.diku.dk/~pisinger/codes.html)。数据文件的格式如下：

- 第一行：预订请求的数量
- 第二行：

- 行程总车站数
- 火车的座位容量
- 接下来的每一行，对应每个请求：

- 行程经过的车站数
- 座位数
- 起始车站
- （两个其他未使用的度量）


## 模型

团体座位预订背包问题 (GSR-KP) 的Hexaly模型使用 [optional interval decision](https://www.hexaly.com/docs/last/modelerreference/standardlibrary/builtinfunctions.html#optionalInterval) variables 来表示分配给每个请求的连续座位。我们使用 [presence](https://www.hexaly.com/docs/last/modelerreference/standardlibrary/builtinfunctions.html#presence) 算子来检测某个请求是否被选中。

对于每个被选中的请求，对应 interval 的长度等于所需的座位数。

我们确保任意两个在行程上有重叠的被选中请求所分配的座位互不重叠。

目标是最大化火车的总体利用率。模型会计算最优利用率的一个简单上界，并使用 [hxObjectiveThreshold](https://www.hexaly.com/docs/last/modelerreference/standardlibrary/builtinfunctions.html#hxObjectiveThreshold) 参数在达到该上界时停止搜索。


## Python 实现


In [ ]:
# Copyright (c) Hexaly. Permission is hereby granted to use, copy,
# and modify this code for applications developed with Hexaly.
import hexaly.optimizer
import sys

if len(sys.argv) < 2:
    print("Usage: python group_seat_reservation_kp.py inputFile [outputFile] [timeLimit]")
    sys.exit(1)

def read_integers(filename):
    with open(filename) as f:
        return [int(elem) for elem in f.read().split()]

with hexaly.optimizer.HexalyOptimizer() as optimizer:
    # Read instance data
    file_it = iter(read_integers(sys.argv[1]))

    nbRequests = int(next(file_it))
    nbStations = int(next(file_it))
    nbSeats = int(next(file_it))

    nbStationsRequest = [0] * nbRequests
    nbSeatsRequest = [0] * nbRequests
    firstStationRequest = [0] * nbRequests
    for i in range(nbRequests):
        nbStationsRequest[i] = int(next(file_it))
        nbSeatsRequest[i] = int(next(file_it))
        firstStationRequest[i] = int(next(file_it))
        int(next(file_it))  # skip value
        int(next(file_it))  # skip value

    # Requests that share at least one station
    sharingStation = [[] for _ in range(nbRequests)]
    for i in range(nbRequests):
        for j in range(i + 1, nbRequests):
            if (firstStationRequest[i] < firstStationRequest[j] + nbStationsRequest[j]
                    and firstStationRequest[i] + nbStationsRequest[i] > firstStationRequest[j]):
                sharingStation[i].append(j)

    # Declare the optimization model
    model = optimizer.model

    # OptionalInterval: requestSelected[i] represents the seats occupied by the reservation i
    requestSelected = [model.optional_interval(0, nbSeats) for _ in range(nbRequests)]

    # If request i is selected then we allocate the corresponding number of seats
    for i in range(nbRequests):
        model.constraint(
                model.iif(
                model.presence(requestSelected[i]),
                model.length(requestSelected[i]) == nbSeatsRequest[i],
                True))

    # If two requests overlap on any station segment, they must not use the same seats
    for i in range(nbRequests):
        for j in sharingStation[i]:
            model.constraint(model.iif(
                    model.and_(model.presence(requestSelected[i]),
                    model.presence(requestSelected[j])),
                    model.or_(requestSelected[j] < requestSelected[i],
                    requestSelected[i] < requestSelected[j]),
                    True))

    # Compute the utilization of the train
    utilization = model.sum([model.presence(requestSelected[i])
            * nbSeatsRequest[i]
            * nbStationsRequest[i]
            for i in range(nbRequests)])

    # Objective: Maximize the utilization of the train
    model.maximize(utilization)

    model.close()

    # Parameterize the optimizer
    if len(sys.argv) >= 4:
        optimizer.param.time_limit = int(sys.argv[3])
    else:
        optimizer.param.time_limit = 5

    # Stop the search if the upper threshold is reached
    optimizer.param.set_objective_threshold(0, nbSeats * nbStations)

    optimizer.solve()

    # Write the solution in a file using the following format:
    #  - 1st line: total utilization of the train
    #  - For each selected request: which seats are occupied
    if len(sys.argv) >= 3:
        with open(sys.argv[2], "w") as f:
            f.write("%d\n" % utilization.value)
            for i in range(nbRequests):
                seats = requestSelected[i].value
                if not seats.is_void():
                    f.write(str(i) + ": [" + str(seats.start())
                            + "..." + str(seats.end() - 1) + "]\n")
